# Software FJ – Fase 4: Manejo de excepciones en POO
**Programación 213023 – UNAD** · Alejandro Rincón García (entrega individual)

Sistema de gestión de clientes, servicios y reservas sin base de datos. Ejecute las celdas en orden; la última corre las 12 simulaciones y muestra el archivo de logs.

## 1. Excepciones personalizadas

In [ ]:
"""
excepciones.py
Excepciones personalizadas del sistema Software FJ.

Se define una jerarquia propia de excepciones para que cada tipo de error
del negocio pueda capturarse de forma especifica, registrarse en el log
y permitir que la aplicacion siga funcionando.
"""


# Clase base de todas las excepciones del sistema; hereda de Exception
class SoftwareFJError(Exception):
    """Excepcion raiz del sistema Software FJ."""

    # Constructor: recibe el mensaje y un codigo opcional de error
    def __init__(self, mensaje, codigo="FJ-000"):
        # Llama al constructor de Exception para guardar el mensaje
        super().__init__(mensaje)
        # Guarda el codigo de error para identificarlo en los logs
        self.codigo = codigo

    # Representacion en texto: incluye el codigo y el mensaje
    def __str__(self):
        # Devuelve el texto con formato [codigo] mensaje
        return f"[{self.codigo}] {super().__str__()}"


# Error de validacion de datos (cliente, servicio o reserva con datos invalidos)
class DatosInvalidosError(SoftwareFJError):
    """Se lanza cuando un dato no cumple las reglas de validacion."""

    # Constructor: recibe el campo que fallo y el motivo
    def __init__(self, campo, motivo):
        # Guarda el nombre del campo invalido
        self.campo = campo
        # Construye el mensaje y asigna el codigo FJ-100
        super().__init__(f"Dato invalido en '{campo}': {motivo}", "FJ-100")


# Error cuando falta un parametro obligatorio
class ParametroFaltanteError(SoftwareFJError):
    """Se lanza cuando no se envia un parametro requerido."""

    # Constructor: recibe el nombre del parametro faltante
    def __init__(self, parametro):
        # Construye el mensaje y asigna el codigo FJ-110
        super().__init__(f"Falta el parametro obligatorio '{parametro}'", "FJ-110")


# Error cuando un servicio no esta disponible para reservar
class ServicioNoDisponibleError(SoftwareFJError):
    """Se lanza cuando se intenta reservar un servicio inactivo u ocupado."""

    # Constructor: recibe el nombre del servicio
    def __init__(self, nombre_servicio):
        # Construye el mensaje y asigna el codigo FJ-200
        super().__init__(f"El servicio '{nombre_servicio}' no esta disponible", "FJ-200")


# Error de reserva (estado incorrecto, duracion invalida, etc.)
class ReservaInvalidaError(SoftwareFJError):
    """Se lanza cuando una reserva no puede crearse o procesarse."""

    # Constructor: recibe el motivo del fallo
    def __init__(self, motivo):
        # Construye el mensaje y asigna el codigo FJ-300
        super().__init__(f"Reserva invalida: {motivo}", "FJ-300")


# Error de operacion no permitida (p. ej. cancelar una reserva ya cancelada)
class OperacionNoPermitidaError(SoftwareFJError):
    """Se lanza cuando se intenta una operacion que el estado actual no permite."""

    # Constructor: recibe la operacion y el estado actual
    def __init__(self, operacion, estado):
        # Construye el mensaje y asigna el codigo FJ-310
        super().__init__(
            f"No se permite '{operacion}' cuando la reserva esta en estado '{estado}'",
            "FJ-310",
        )


# Error de calculo (costos negativos, impuestos fuera de rango, etc.)
class CalculoInconsistenteError(SoftwareFJError):
    """Se lanza cuando un calculo de costos produce un resultado inconsistente."""

    # Constructor: recibe el detalle del problema de calculo
    def __init__(self, detalle):
        # Construye el mensaje y asigna el codigo FJ-400
        super().__init__(f"Calculo inconsistente: {detalle}", "FJ-400")


## 2. Configuración del archivo de logs

In [ ]:
"""
registro_logs.py
Configuracion del archivo de logs del sistema Software FJ.

Todos los eventos (INFO) y errores (ERROR/WARNING) se escriben en el archivo
'software_fj.log'. No se usa base de datos: el archivo de logs es el unico
almacenamiento en disco permitido por el enunciado.
"""

# Importa el modulo estandar de logging de Python
import logging
# Importa os para construir la ruta del archivo de logs
import os

# Ruta del archivo de logs: carpeta raiz del proyecto
RUTA_LOG = "software_fj.log"  # en el notebook el log se crea en la carpeta actual


# Funcion que crea (una sola vez) y devuelve el logger del sistema
def obtener_logger(nombre="SoftwareFJ"):
    """Devuelve un logger configurado para escribir en archivo y consola."""
    # Obtiene (o crea) el logger con el nombre indicado
    logger = logging.getLogger(nombre)
    # Si ya tiene manejadores, se devuelve tal cual para no duplicar lineas
    if logger.handlers:
        return logger
    # Nivel minimo que se registrara: INFO (eventos) y superiores (errores)
    logger.setLevel(logging.INFO)
    # Formato de cada linea: fecha, nivel, modulo y mensaje
    formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(module)s | %(message)s")
    # Manejador que escribe en el archivo de logs con codificacion UTF-8
    manejador_archivo = logging.FileHandler(RUTA_LOG, mode="a", encoding="utf-8")
    # Asigna el formato al manejador de archivo
    manejador_archivo.setFormatter(formato)
    # Agrega el manejador de archivo al logger
    logger.addHandler(manejador_archivo)
    # Evita que el mensaje se propague al logger raiz (no se duplica en consola)
    logger.propagate = False
    # Devuelve el logger listo para usar
    return logger


## 3. Clase abstracta EntidadBase y clase Cliente

In [ ]:
"""
entidades.py
Clase abstracta EntidadBase y clase Cliente del sistema Software FJ.
"""

# Importa ABC y abstractmethod para definir clases abstractas
from abc import ABC, abstractmethod
# Importa re para validar correos y telefonos con expresiones regulares
import re
# Importa itertools.count para generar identificadores consecutivos
from itertools import count

# Importa las excepciones personalizadas que usa este modulo
# Importa la funcion que entrega el logger configurado

# Logger del modulo para registrar eventos y errores
log = obtener_logger()


# Clase abstracta que representa cualquier entidad general del sistema
class EntidadBase(ABC):
    """Entidad general: todo objeto del sistema tiene un id y sabe describirse y validarse."""

    # Contador compartido para asignar ids unicos a cada entidad
    _contador_ids = count(1)

    # Constructor comun de todas las entidades
    def __init__(self):
        # Asigna un identificador unico y privado a la entidad
        self.__id = next(EntidadBase._contador_ids)

    # Propiedad de solo lectura para consultar el id (encapsulacion)
    @property
    def id(self):
        # Devuelve el id privado
        return self.__id

    # Metodo abstracto: cada entidad debe definir como se describe
    @abstractmethod
    def describir(self):
        """Devuelve una descripcion en texto de la entidad."""

    # Metodo abstracto: cada entidad debe definir sus validaciones
    @abstractmethod
    def validar(self):
        """Valida el estado interno de la entidad y lanza excepcion si es invalido."""


# Clase Cliente: hereda de EntidadBase y encapsula los datos personales
class Cliente(EntidadBase):
    """Cliente de Software FJ con validaciones robustas de sus datos."""

    # Patron para validar correos electronicos basicos
    _PATRON_CORREO = re.compile(r"^[\w\.-]+@[\w\.-]+\.\w{2,}$")
    # Patron para validar telefonos de 7 a 10 digitos
    _PATRON_TELEFONO = re.compile(r"^\d{7,10}$")

    # Constructor: recibe documento, nombre, correo y telefono
    def __init__(self, documento=None, nombre=None, correo=None, telefono=None):
        # Inicializa la parte comun (id) de la entidad
        super().__init__()
        # Asigna cada dato a traves de su setter para que se valide
        self.documento = documento
        self.nombre = nombre
        self.correo = correo
        self.telefono = telefono
        # Valida el objeto completo una vez asignados los datos
        self.validar()
        # Registra en el log que el cliente se creo correctamente
        log.info(f"Cliente registrado: {self.nombre} (doc {self.documento})")

    # ----- Propiedades con validacion (encapsulacion de datos personales) -----

    # Getter del documento
    @property
    def documento(self):
        # Devuelve el documento almacenado en el atributo privado
        return self.__documento

    # Setter del documento: obligatorio y solo digitos (6 a 10)
    @documento.setter
    def documento(self, valor):
        # Si no se envio el documento se lanza ParametroFaltanteError
        if valor is None or str(valor).strip() == "":
            raise ParametroFaltanteError("documento")
        # Convierte a texto y quita espacios
        valor = str(valor).strip()
        # Verifica que solo tenga digitos y longitud valida
        if not valor.isdigit() or not 6 <= len(valor) <= 10:
            raise DatosInvalidosError("documento", "debe tener entre 6 y 10 digitos")
        # Guarda el valor en el atributo privado
        self.__documento = valor

    # Getter del nombre
    @property
    def nombre(self):
        # Devuelve el nombre privado
        return self.__nombre

    # Setter del nombre: obligatorio, minimo 3 letras, sin numeros
    @nombre.setter
    def nombre(self, valor):
        # Verifica que el nombre exista
        if valor is None or str(valor).strip() == "":
            raise ParametroFaltanteError("nombre")
        # Quita espacios al inicio y al final
        valor = str(valor).strip()
        # Verifica longitud minima y que no contenga numeros
        if len(valor) < 3 or any(c.isdigit() for c in valor):
            raise DatosInvalidosError("nombre", "minimo 3 caracteres y sin numeros")
        # Guarda el nombre en formato titulo
        self.__nombre = valor.title()

    # Getter del correo
    @property
    def correo(self):
        # Devuelve el correo privado
        return self.__correo

    # Setter del correo: obligatorio y con formato valido
    @correo.setter
    def correo(self, valor):
        # Verifica que el correo exista
        if not valor:
            raise ParametroFaltanteError("correo")
        # Verifica el formato con la expresion regular
        if not Cliente._PATRON_CORREO.match(str(valor)):
            raise DatosInvalidosError("correo", f"formato no valido ({valor})")
        # Guarda el correo en minusculas
        self.__correo = str(valor).lower()

    # Getter del telefono
    @property
    def telefono(self):
        # Devuelve el telefono privado
        return self.__telefono

    # Setter del telefono: opcional, pero si se envia debe ser valido
    @telefono.setter
    def telefono(self, valor):
        # Si no se envia telefono se guarda None (es opcional)
        if valor in (None, ""):
            self.__telefono = None
            return
        # Verifica que tenga entre 7 y 10 digitos
        if not Cliente._PATRON_TELEFONO.match(str(valor)):
            raise DatosInvalidosError("telefono", "debe tener entre 7 y 10 digitos")
        # Guarda el telefono como texto
        self.__telefono = str(valor)

    # Implementacion del metodo abstracto validar
    def validar(self):
        # Los setters ya validan cada campo; aqui se verifica que los obligatorios existan
        for campo in ("documento", "nombre", "correo"):
            # Si algun campo obligatorio quedo vacio se lanza la excepcion
            if not getattr(self, campo, None):
                raise ParametroFaltanteError(campo)
        # Si todo esta bien devuelve True
        return True

    # Implementacion del metodo abstracto describir
    def describir(self):
        # Devuelve una descripcion corta del cliente (sin exponer el telefono)
        return f"Cliente #{self.id}: {self.nombre} - {self.correo}"

    # Representacion del objeto en texto
    def __str__(self):
        # Reutiliza el metodo describir
        return self.describir()


## 4. Clase abstracta Servicio y servicios especializados

In [ ]:
"""
servicios.py
Clase abstracta Servicio y tres servicios especializados:
ReservaSala, AlquilerEquipo y AsesoriaEspecializada.

Cada servicio sobrescribe (polimorfismo) los metodos para calcular costos,
describirse y validar parametros. El metodo calcular_costo esta "sobrecargado":
Python no permite varias firmas con el mismo nombre, asi que la sobrecarga se
implementa con parametros opcionales (impuesto, descuento) y argumentos por nombre.
"""

# Importa ABC y abstractmethod para la clase abstracta Servicio
from abc import abstractmethod

# Importa la entidad base abstracta
# Importa las excepciones personalizadas que se usan en los servicios
# Importa el logger del sistema

# Logger del modulo
log = obtener_logger()


# Clase abstracta Servicio: hereda de EntidadBase
class Servicio(EntidadBase):
    """Servicio generico ofrecido por Software FJ."""

    # Impuesto por defecto (IVA 19 %)
    IVA = 0.19

    # Constructor comun de todos los servicios
    def __init__(self, nombre, tarifa_hora):
        # Inicializa el id de la entidad
        super().__init__()
        # Valida que el nombre exista
        if not nombre:
            raise ParametroFaltanteError("nombre del servicio")
        # Valida que la tarifa sea un numero positivo
        if not isinstance(tarifa_hora, (int, float)) or tarifa_hora <= 0:
            raise DatosInvalidosError("tarifa_hora", "debe ser un numero mayor que cero")
        # Atributos protegidos del servicio
        self._nombre = nombre
        self._tarifa_hora = float(tarifa_hora)
        # Todo servicio inicia disponible
        self._disponible = True

    # Propiedad de solo lectura para el nombre
    @property
    def nombre(self):
        # Devuelve el nombre del servicio
        return self._nombre

    # Propiedad para saber si el servicio esta disponible
    @property
    def disponible(self):
        # Devuelve el estado de disponibilidad
        return self._disponible

    # Marca el servicio como no disponible (p. ej. mantenimiento)
    def deshabilitar(self):
        # Cambia la disponibilidad a False
        self._disponible = False
        # Registra el evento
        log.warning(f"Servicio deshabilitado: {self._nombre}")

    # Marca el servicio como disponible
    def habilitar(self):
        # Cambia la disponibilidad a True
        self._disponible = True
        # Registra el evento
        log.info(f"Servicio habilitado: {self._nombre}")

    # Verifica disponibilidad y lanza excepcion si no lo esta
    def verificar_disponibilidad(self):
        # Si no esta disponible se lanza ServicioNoDisponibleError
        if not self._disponible:
            raise ServicioNoDisponibleError(self._nombre)
        # Si esta disponible devuelve True
        return True

    # Metodo abstracto: costo base segun la duracion (cada servicio lo calcula distinto)
    @abstractmethod
    def costo_base(self, horas):
        """Calcula el costo base del servicio para la cantidad de horas indicada."""

    # Metodo abstracto: validacion de parametros propios de cada servicio
    @abstractmethod
    def validar_parametros(self, horas):
        """Valida los parametros de uso del servicio (horas, capacidad, etc.)."""

    # Implementacion general de validar (de EntidadBase)
    def validar(self):
        # Un servicio es valido si tiene nombre y tarifa positiva
        return bool(self._nombre) and self._tarifa_hora > 0

    # Metodo "sobrecargado": admite 3 variantes de llamada
    #   calcular_costo(horas)                       -> costo con IVA
    #   calcular_costo(horas, impuesto=0.0)         -> costo con impuesto personalizado
    #   calcular_costo(horas, descuento=0.10)       -> costo con descuento
    #   calcular_costo(horas, impuesto=x, descuento=y) -> ambas
    def calcular_costo(self, horas, impuesto=None, descuento=None):
        """Calcula el costo total aplicando impuesto y/o descuento opcionales."""
        # Valida los parametros propios del servicio (polimorfismo)
        self.validar_parametros(horas)
        # Si no se envia impuesto se usa el IVA por defecto
        impuesto = Servicio.IVA if impuesto is None else impuesto
        # Si no se envia descuento se usa 0
        descuento = 0.0 if descuento is None else descuento
        # El impuesto debe estar entre 0 y 1
        if not 0 <= impuesto <= 1:
            raise CalculoInconsistenteError(f"impuesto fuera de rango ({impuesto})")
        # El descuento debe estar entre 0 y 0.5 (maximo 50 %)
        if not 0 <= descuento <= 0.5:
            raise CalculoInconsistenteError(f"descuento fuera de rango ({descuento})")
        # Calcula el costo base usando la implementacion de la subclase
        base = self.costo_base(horas)
        # Aplica el descuento y luego el impuesto
        total = base * (1 - descuento) * (1 + impuesto)
        # Un total negativo o cero indica un calculo inconsistente
        if total <= 0:
            raise CalculoInconsistenteError(f"total no positivo ({total})")
        # Devuelve el total redondeado a 2 decimales
        return round(total, 2)

    # Representacion en texto del servicio
    def __str__(self):
        # Reutiliza describir (polimorfico)
        return self.describir()


# Servicio especializado 1: reserva de salas
class ReservaSala(Servicio):
    """Reserva de salas de reuniones con capacidad maxima."""

    # Constructor: agrega la capacidad de la sala
    def __init__(self, nombre, tarifa_hora, capacidad):
        # Inicializa la parte comun del servicio
        super().__init__(nombre, tarifa_hora)
        # Valida la capacidad (entero positivo)
        if not isinstance(capacidad, int) or capacidad <= 0:
            raise DatosInvalidosError("capacidad", "debe ser un entero positivo")
        # Guarda la capacidad
        self._capacidad = capacidad
        # Registra el evento
        log.info(f"Servicio creado: {self.describir()}")

    # Sobrescribe costo_base: tarifa por hora, con recargo si supera 8 horas
    def costo_base(self, horas):
        # Costo normal por horas
        costo = self._tarifa_hora * horas
        # Recargo del 10 % para reservas de mas de 8 horas
        return costo * 1.10 if horas > 8 else costo

    # Sobrescribe validar_parametros: entre 1 y 12 horas
    def validar_parametros(self, horas):
        # Verifica tipo y rango de las horas
        if not isinstance(horas, (int, float)) or not 1 <= horas <= 12:
            raise DatosInvalidosError("horas", "una sala se reserva entre 1 y 12 horas")
        # Devuelve True si es valido
        return True

    # Sobrescribe describir
    def describir(self):
        # Descripcion particular de la sala
        return f"Sala '{self._nombre}' (cap. {self._capacidad}) - ${self._tarifa_hora:,.0f}/h"


# Servicio especializado 2: alquiler de equipos
class AlquilerEquipo(Servicio):
    """Alquiler de equipos (portatiles, proyectores) con deposito fijo."""

    # Constructor: agrega el tipo de equipo y el deposito
    def __init__(self, nombre, tarifa_hora, tipo_equipo, deposito=50000):
        # Inicializa la parte comun
        super().__init__(nombre, tarifa_hora)
        # Valida que se indique el tipo de equipo
        if not tipo_equipo:
            raise ParametroFaltanteError("tipo_equipo")
        # Valida que el deposito no sea negativo
        if deposito < 0:
            raise DatosInvalidosError("deposito", "no puede ser negativo")
        # Guarda los atributos propios
        self._tipo_equipo = tipo_equipo
        self._deposito = deposito
        # Registra el evento
        log.info(f"Servicio creado: {self.describir()}")

    # Sobrescribe costo_base: tarifa por hora mas deposito fijo
    def costo_base(self, horas):
        # Suma el deposito al costo por horas
        return self._tarifa_hora * horas + self._deposito

    # Sobrescribe validar_parametros: entre 1 y 72 horas
    def validar_parametros(self, horas):
        # Verifica tipo y rango
        if not isinstance(horas, (int, float)) or not 1 <= horas <= 72:
            raise DatosInvalidosError("horas", "un equipo se alquila entre 1 y 72 horas")
        # Devuelve True si es valido
        return True

    # Sobrescribe describir
    def describir(self):
        # Descripcion particular del equipo
        return f"Equipo '{self._nombre}' ({self._tipo_equipo}) - ${self._tarifa_hora:,.0f}/h + deposito"


# Servicio especializado 3: asesorias especializadas
class AsesoriaEspecializada(Servicio):
    """Asesoria con un especialista; el nivel define un multiplicador de tarifa."""

    # Multiplicadores de tarifa segun el nivel del asesor
    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}

    # Constructor: agrega el area y el nivel del asesor
    def __init__(self, nombre, tarifa_hora, area, nivel="junior"):
        # Inicializa la parte comun
        super().__init__(nombre, tarifa_hora)
        # Valida que el area exista
        if not area:
            raise ParametroFaltanteError("area")
        # Valida que el nivel sea uno de los permitidos
        if nivel not in AsesoriaEspecializada.NIVELES:
            raise DatosInvalidosError("nivel", f"debe ser uno de {list(self.NIVELES)}")
        # Guarda los atributos propios
        self._area = area
        self._nivel = nivel
        # Registra el evento
        log.info(f"Servicio creado: {self.describir()}")

    # Sobrescribe costo_base: tarifa x horas x multiplicador del nivel
    def costo_base(self, horas):
        # Aplica el multiplicador del nivel
        return self._tarifa_hora * horas * AsesoriaEspecializada.NIVELES[self._nivel]

    # Sobrescribe validar_parametros: entre 1 y 6 horas por sesion
    def validar_parametros(self, horas):
        # Verifica tipo y rango
        if not isinstance(horas, (int, float)) or not 1 <= horas <= 6:
            raise DatosInvalidosError("horas", "una asesoria dura entre 1 y 6 horas")
        # Devuelve True si es valido
        return True

    # Sobrescribe describir
    def describir(self):
        # Descripcion particular de la asesoria
        return f"Asesoria '{self._nombre}' en {self._area} (nivel {self._nivel}) - ${self._tarifa_hora:,.0f}/h"


## 5. Clase Reserva (try/except/else/finally y encadenamiento)

In [ ]:
"""
reserva.py
Clase Reserva: integra cliente, servicio, duracion y estado.
Implementa confirmacion, cancelacion y procesamiento con manejo de excepciones
(try/except, try/except/else, try/except/finally y encadenamiento con 'raise ... from').
"""

# Importa la entidad base abstracta
# Importa la clase abstracta Servicio para validar el tipo
# Importa las excepciones personalizadas
# Importa el logger

# Logger del modulo
log = obtener_logger()


# Clase Reserva: hereda de EntidadBase
class Reserva(EntidadBase):
    """Reserva de un servicio por parte de un cliente."""

    # Estados posibles de una reserva
    PENDIENTE, CONFIRMADA, CANCELADA, PROCESADA = "PENDIENTE", "CONFIRMADA", "CANCELADA", "PROCESADA"

    # Constructor: recibe cliente, servicio y duracion en horas
    def __init__(self, cliente, servicio, horas):
        # Inicializa el id
        super().__init__()
        # El cliente debe ser una instancia de Cliente
        if not isinstance(cliente, Cliente):
            raise ReservaInvalidaError("el cliente no es valido")
        # El servicio debe ser una instancia de Servicio
        if not isinstance(servicio, Servicio):
            raise ReservaInvalidaError("el servicio no es valido")
        # Guarda los atributos privados
        self.__cliente = cliente
        self.__servicio = servicio
        self.__horas = horas
        # Estado inicial
        self.__estado = Reserva.PENDIENTE
        # Costo aun no calculado
        self.__costo = None
        # Valida la reserva completa
        self.validar()
        # Registra el evento de creacion
        log.info(f"Reserva #{self.id} creada: {cliente.nombre} -> {servicio.nombre} ({horas} h)")

    # Propiedad de solo lectura del estado
    @property
    def estado(self):
        # Devuelve el estado actual
        return self.__estado

    # Propiedad de solo lectura del costo
    @property
    def costo(self):
        # Devuelve el costo calculado (o None)
        return self.__costo

    # Validacion de la reserva (metodo abstracto de EntidadBase)
    def validar(self):
        # Se intenta validar las horas con las reglas del servicio (polimorfismo)
        try:
            # Cada servicio valida sus propios parametros
            self.__servicio.validar_parametros(self.__horas)
        # Si las horas no son validas se captura el error del servicio...
        except DatosInvalidosError as error:
            # ...y se encadena en un error de reserva (encadenamiento de excepciones)
            raise ReservaInvalidaError(f"duracion no permitida para {self.__servicio.nombre}") from error
        # Si no hubo error, la reserva es valida
        return True

    # Confirma la reserva: usa try/except/else
    def confirmar(self, impuesto=None, descuento=None):
        """Confirma la reserva calculando su costo."""
        # Solo se puede confirmar una reserva pendiente
        if self.__estado != Reserva.PENDIENTE:
            raise OperacionNoPermitidaError("confirmar", self.__estado)
        try:
            # Verifica que el servicio este disponible
            self.__servicio.verificar_disponibilidad()
            # Calcula el costo (metodo sobrecargado del servicio)
            costo = self.__servicio.calcular_costo(self.__horas, impuesto=impuesto, descuento=descuento)
        except CalculoInconsistenteError as error:
            # Error de calculo: se registra y se encadena como reserva invalida
            log.error(f"Reserva #{self.id}: error de calculo -> {error}")
            raise ReservaInvalidaError("no fue posible calcular el costo") from error
        else:
            # Solo si no hubo errores: se guarda el costo y se cambia el estado
            self.__costo = costo
            self.__estado = Reserva.CONFIRMADA
            # Registra la confirmacion
            log.info(f"Reserva #{self.id} CONFIRMADA. Costo: ${costo:,.2f}")
            # Devuelve el costo calculado
            return costo

    # Cancela la reserva: usa try/except/finally
    def cancelar(self, motivo="sin motivo"):
        """Cancela la reserva si su estado lo permite."""
        try:
            # No se puede cancelar una reserva ya cancelada o procesada
            if self.__estado in (Reserva.CANCELADA, Reserva.PROCESADA):
                raise OperacionNoPermitidaError("cancelar", self.__estado)
            # Cambia el estado a cancelada
            self.__estado = Reserva.CANCELADA
            # Registra la cancelacion
            log.info(f"Reserva #{self.id} CANCELADA. Motivo: {motivo}")
        except OperacionNoPermitidaError:
            # Registra el intento invalido y relanza la excepcion
            log.error(f"Reserva #{self.id}: intento de cancelacion no permitido (estado {self.__estado})")
            raise
        finally:
            # Siempre se deja traza de la operacion, haya o no error
            log.info(f"Reserva #{self.id}: fin del intento de cancelacion")

    # Procesa la reserva (pago/ejecucion): usa try/except/else/finally
    def procesar(self):
        """Procesa una reserva confirmada."""
        try:
            # Solo se procesan reservas confirmadas
            if self.__estado != Reserva.CONFIRMADA:
                raise OperacionNoPermitidaError("procesar", self.__estado)
            # Se verifica de nuevo la disponibilidad del servicio al momento de procesar
            self.__servicio.verificar_disponibilidad()
        except SoftwareFJError as error:
            # Cualquier error del negocio se registra y se encadena
            log.error(f"Reserva #{self.id}: no se pudo procesar -> {error}")
            raise ReservaInvalidaError(f"la reserva #{self.id} no pudo procesarse") from error
        else:
            # Si todo salio bien, se marca como procesada
            self.__estado = Reserva.PROCESADA
            # Registra el exito
            log.info(f"Reserva #{self.id} PROCESADA correctamente")
        finally:
            # Siempre se registra el cierre del intento
            log.info(f"Reserva #{self.id}: fin del procesamiento (estado {self.__estado})")

    # Descripcion de la reserva
    def describir(self):
        # Texto con el cliente, servicio, horas, estado y costo
        costo = f"${self.__costo:,.2f}" if self.__costo is not None else "sin calcular"
        return (f"Reserva #{self.id}: {self.__cliente.nombre} | {self.__servicio.nombre} | "
                f"{self.__horas} h | {self.__estado} | {costo}")

    # Representacion en texto
    def __str__(self):
        # Reutiliza describir
        return self.describir()


## 6. Gestor del sistema (listas internas)

In [ ]:
"""
sistema.py
Clase SistemaFJ: administra clientes, servicios y reservas en listas internas
(sin base de datos). Cada operacion captura sus excepciones, las registra en el
log y devuelve None para que la aplicacion siga funcionando.
"""

# Importa las clases del dominio
# Importa la excepcion raiz para capturar cualquier error del negocio
# Importa el logger

# Logger del modulo
log = obtener_logger()


# Clase que coordina toda la operacion de Software FJ
class SistemaFJ:
    """Gestor central de clientes, servicios y reservas."""

    # Constructor: crea las listas internas vacias
    def __init__(self):
        # Lista interna de clientes
        self.clientes = []
        # Lista interna de servicios
        self.servicios = []
        # Lista interna de reservas
        self.reservas = []
        # Contadores de operaciones exitosas y fallidas
        self.exitos = 0
        self.fallos = 0

    # Metodo privado que ejecuta cualquier operacion de forma segura
    def _ejecutar(self, descripcion, funcion, *args, **kwargs):
        """Ejecuta 'funcion' capturando errores para que el sistema nunca se detenga."""
        try:
            # Intenta ejecutar la operacion solicitada
            resultado = funcion(*args, **kwargs)
        except SoftwareFJError as error:
            # Error controlado del negocio: se registra con su causa (si fue encadenado)
            causa = f" | causa: {error.__cause__}" if error.__cause__ else ""
            log.error(f"{descripcion} FALLIDA -> {error}{causa}")
            # Se informa en consola y se cuenta el fallo
            print(f"   [X] {descripcion}: {error}{causa}")
            self.fallos += 1
            # Devuelve None para indicar fallo sin detener la app
            return None
        except Exception as error:
            # Error inesperado: tambien se registra con traza completa
            log.exception(f"{descripcion} ERROR INESPERADO -> {error!r}")
            print(f"   [!] {descripcion}: error inesperado {error!r}")
            self.fallos += 1
            return None
        else:
            # Solo si no hubo error se cuenta el exito
            self.exitos += 1
            print(f"   [OK] {descripcion}")
            # Devuelve el resultado de la operacion
            return resultado

    # Registra un cliente y lo agrega a la lista interna
    def registrar_cliente(self, **datos):
        # Ejecuta la creacion del cliente de forma segura
        cliente = self._ejecutar(f"Registrar cliente {datos.get('nombre')}", Cliente, **datos)
        # Si se creo correctamente se agrega a la lista
        if cliente:
            self.clientes.append(cliente)
        # Devuelve el cliente o None
        return cliente

    # Crea un servicio de la clase indicada y lo agrega a la lista
    def crear_servicio(self, clase, *args, **kwargs):
        # Ejecuta la creacion del servicio de forma segura
        servicio = self._ejecutar(f"Crear servicio {clase.__name__} '{args[0] if args else '?'}'",
                                  clase, *args, **kwargs)
        # Si se creo correctamente se agrega a la lista
        if servicio:
            self.servicios.append(servicio)
        # Devuelve el servicio o None
        return servicio

    # Crea una reserva y la agrega a la lista
    def crear_reserva(self, cliente, servicio, horas):
        # Nombre legible para el log aunque cliente o servicio sean None
        nombre_c = getattr(cliente, "nombre", cliente)
        nombre_s = getattr(servicio, "nombre", servicio)
        # Ejecuta la creacion de la reserva de forma segura
        reserva = self._ejecutar(f"Crear reserva {nombre_c} -> {nombre_s} ({horas} h)",
                                 Reserva, cliente, servicio, horas)
        # Si se creo correctamente se agrega a la lista
        if reserva:
            self.reservas.append(reserva)
        # Devuelve la reserva o None
        return reserva

    # Confirma una reserva existente (admite impuesto y descuento opcionales)
    def confirmar_reserva(self, reserva, **kwargs):
        # Si la reserva no existe se registra el fallo sin detener la app
        if reserva is None:
            log.warning("Se intento confirmar una reserva inexistente")
            print("   [X] Confirmar reserva: la reserva no existe")
            self.fallos += 1
            return None
        # Ejecuta la confirmacion de forma segura
        return self._ejecutar(f"Confirmar reserva #{reserva.id}", reserva.confirmar, **kwargs)

    # Cancela una reserva existente
    def cancelar_reserva(self, reserva, motivo="solicitud del cliente"):
        # Ejecuta la cancelacion de forma segura
        return self._ejecutar(f"Cancelar reserva #{reserva.id}", reserva.cancelar, motivo)

    # Procesa una reserva existente
    def procesar_reserva(self, reserva):
        # Ejecuta el procesamiento de forma segura
        return self._ejecutar(f"Procesar reserva #{reserva.id}", reserva.procesar)

    # Muestra un resumen del estado del sistema
    def resumen(self):
        # Imprime totales y el detalle de cada reserva
        print("\n================ RESUMEN DEL SISTEMA ================")
        print(f"Clientes registrados: {len(self.clientes)}")
        print(f"Servicios creados:    {len(self.servicios)}")
        print(f"Reservas creadas:     {len(self.reservas)}")
        print(f"Operaciones exitosas: {self.exitos} | fallidas (controladas): {self.fallos}")
        # Recorre la lista de reservas y muestra cada una (polimorfismo de describir)
        for reserva in self.reservas:
            print("  -", reserva)
        # Registra el resumen en el log
        log.info(f"Resumen: {self.exitos} exitos, {self.fallos} fallos controlados")


## 7. Programa principal: 12 simulaciones

In [ ]:
"""
main.py
Programa principal de Software FJ - Fase 4 (Programacion 213023, UNAD).

Simula mas de 10 operaciones completas (validas e invalidas) para demostrar que
el sistema continua funcionando aunque ocurran errores. Todos los eventos y
errores quedan registrados en 'software_fj.log'.

Ejecucion:  python main.py
"""

# Importa el gestor del sistema
# Importa los tres servicios especializados
# Importa la ruta del log para mostrarla al final

# Logger principal
log = obtener_logger()


# Funcion auxiliar para imprimir el titulo de cada simulacion
def titulo(numero, texto):
    # Imprime el encabezado de la simulacion
    print(f"\n--- Simulacion {numero}: {texto} ---")
    # Registra el inicio de la simulacion en el log
    log.info(f"=== Simulacion {numero}: {texto} ===")


# Funcion principal que ejecuta todas las simulaciones
def main():
    # Registra el inicio de la ejecucion
    log.info("========== INICIO DE EJECUCION SOFTWARE FJ ==========")
    # Crea el sistema con sus listas internas vacias
    sistema = SistemaFJ()

    # 1. Registro de clientes validos
    titulo(1, "Registro de clientes validos")
    ana = sistema.registrar_cliente(documento="1023456789", nombre="ana torres",
                                    correo="ana.torres@correo.com", telefono="3104567890")
    luis = sistema.registrar_cliente(documento="80123456", nombre="Luis Pardo",
                                     correo="lpardo@empresa.co")

    # 2. Registro de clientes invalidos (correo mal escrito, documento con letras, nombre faltante)
    titulo(2, "Registro de clientes invalidos")
    sistema.registrar_cliente(documento="12345678", nombre="Carlos Ruiz", correo="carlos-arroba-mail")
    sistema.registrar_cliente(documento="ABC123", nombre="Maria Gil", correo="maria@mail.com")
    sistema.registrar_cliente(documento="99887766", nombre=None, correo="sin@nombre.com")

    # 3. Creacion correcta de los tres tipos de servicio
    titulo(3, "Creacion correcta de servicios")
    sala = sistema.crear_servicio(ReservaSala, "Sala Bogota", 80000, capacidad=12)
    portatil = sistema.crear_servicio(AlquilerEquipo, "Portatil Dell", 15000, tipo_equipo="Portatil")
    asesoria = sistema.crear_servicio(AsesoriaEspecializada, "Asesoria BI", 120000, area="Power BI", nivel="senior")

    # 4. Creacion incorrecta de servicios (tarifa negativa, nivel inexistente, capacidad invalida)
    titulo(4, "Creacion incorrecta de servicios")
    sistema.crear_servicio(ReservaSala, "Sala Fantasma", -5000, capacidad=10)
    sistema.crear_servicio(AsesoriaEspecializada, "Asesoria X", 100000, area="Redes", nivel="maestro")
    sistema.crear_servicio(ReservaSala, "Sala Mini", 50000, capacidad=0)

    # 5. Reserva exitosa con costo por defecto (IVA 19 %) y procesamiento
    titulo(5, "Reserva exitosa: confirmar y procesar")
    r1 = sistema.crear_reserva(ana, sala, 4)
    sistema.confirmar_reserva(r1)
    sistema.procesar_reserva(r1)

    # 6. Reserva con metodo sobrecargado: descuento del 10 % y sin impuesto
    titulo(6, "Costo con descuento y sin impuesto (sobrecarga)")
    r2 = sistema.crear_reserva(luis, asesoria, 2)
    sistema.confirmar_reserva(r2, impuesto=0.0, descuento=0.10)

    # 7. Reserva con duracion invalida (encadenamiento: DatosInvalidos -> ReservaInvalida)
    titulo(7, "Reserva con duracion invalida")
    sistema.crear_reserva(ana, asesoria, 10)

    # 8. Reserva con cliente inexistente (fallo en el registro previo)
    titulo(8, "Reserva con cliente invalido")
    sistema.crear_reserva(None, portatil, 5)

    # 9. Servicio no disponible: se deshabilita y luego se intenta confirmar
    titulo(9, "Servicio no disponible")
    r3 = sistema.crear_reserva(luis, portatil, 24)
    portatil.deshabilitar()
    sistema.confirmar_reserva(r3)
    portatil.habilitar()
    sistema.confirmar_reserva(r3)

    # 10. Calculo inconsistente: descuento del 80 % (fuera de rango)
    titulo(10, "Calculo inconsistente (descuento excesivo)")
    r4 = sistema.crear_reserva(ana, portatil, 8)
    sistema.confirmar_reserva(r4, descuento=0.80)

    # 11. Operaciones no permitidas: cancelar dos veces y procesar una cancelada
    titulo(11, "Operaciones no permitidas sobre una reserva")
    sistema.cancelar_reserva(r4, motivo="cliente desiste")
    sistema.cancelar_reserva(r4)
    sistema.procesar_reserva(r4)

    # 12. Procesar una reserva que no ha sido confirmada y confirmar una inexistente
    titulo(12, "Procesar sin confirmar / confirmar reserva inexistente")
    r5 = sistema.crear_reserva(luis, sala, 9)
    sistema.procesar_reserva(r5)
    sistema.confirmar_reserva(None)
    sistema.confirmar_reserva(r5)
    sistema.procesar_reserva(r5)

    # Muestra el resumen final: la aplicacion llego hasta aqui sin detenerse
    sistema.resumen()
    print(f"\nEl sistema termino sin interrupciones. Revise el log en: {RUTA_LOG}")
    # Registra el fin de la ejecucion
    log.info("========== FIN DE EJECUCION SOFTWARE FJ ==========")


# Punto de entrada: solo se ejecuta si el archivo se corre directamente
if __name__ == "__main__":
    main()


## 8. Contenido del archivo de logs

In [ ]:
# Muestra el log generado por la ejecucion
with open(RUTA_LOG, encoding="utf-8") as f:
    print(f.read())